# 17 — Hypothetical carrying corridors (static, all 20 matches)

**Purpose:** Give H3 a parallel **carrying-route geometric representation**, instead of equating observed `carry=True` to counterfactual carry opportunity.

At each possession start, construct 15 *hypothetical straight segments*: relative attacking-axis angles −60°, −30°, 0°, +30°, +60° at lengths 3, 5 and 10 metres. Distances and headings are **exploratory geometric scenarios**, not calibrated carrying choices, movement forecasts or player-specific feasible actions. Never use the observed `carry` flag, an event outcome, or future frames as a feature.

We test pitch bounds and minimum distances from defenders to the segment and endpoint at **t₀**. Being inside pitch bounds is only a geometric property. Do not label a route *open*, *safe*, *available* or *successful*.

Inputs: `outputs/P1_canonical_decisions_match_*_ROLES.jsonl`, source `data/matches/*/*_match.json`. Output: per-match `outputs/P3_static_carry_corridors_match_*.jsonl` and `outputs/P3_static_carry_corridors_gate.json`.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime,timezone
import json,math

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Run from within SkillCorner repository')
OUT=ROOT/'outputs'; MATCH=ROOT/'data'/'matches'
FILES=sorted(OUT.glob('P1_canonical_decisions_match_*_ROLES.jsonl'))
assert len(FILES)==20,f'Expected 20 canonical files, got {len(FILES)}'
ANGLES=(-60,-30,0,30,60)
LENGTHS=(3.0,5.0,10.0)
assert len(ANGLES)*len(LENGTHS)==15

def finite_xy(obj):
    if obj is None:return None
    if isinstance(obj,(tuple,list)) and len(obj)==2:v=obj
    elif isinstance(obj,dict):v=(obj.get('x'),obj.get('y'))
    else:return None
    try:
        p=tuple(float(z) for z in v)
        return p if all(math.isfinite(z) for z in p) else None
    except (TypeError,ValueError):return None

def point_to_segment(q,a,b):
    dx,dy=b[0]-a[0],b[1]-a[1]
    mag=dx*dx+dy*dy
    assert mag>0
    t=max(0.0,min(1.0,((q[0]-a[0])*dx+(q[1]-a[1])*dy)/mag))
    return math.hypot(q[0]-(a[0]+t*dx),q[1]-(a[1]+t*dy))

def source_signs(meta):
    sides=meta['home_team_side']
    if not isinstance(sides,list) or len(sides)!=2:raise AssertionError('unexpected home_team_side')
    home,away=str(meta['home_team']['id']),str(meta['away_team']['id'])
    signs={}
    for period,label in enumerate(sides,1):
        if label not in ('left_to_right','right_to_left'):raise AssertionError(label)
        s=1 if label=='left_to_right' else -1
        signs[(period,home)]=s;signs[(period,away)]=-s
    return signs


## Compute the corridors

A candidate that extends beyond the pitch is retained with `endpoint_within_pitch=False` rather than silently discarded. Minimum distances are `null` if no defender has finite coordinates; this is **not** interpreted as a clear route. The coordinate system and pitch dimensions come from the source match metadata, not inferred from observed player locations.


In [2]:
TOTAL=Counter(); per_match=[]
for source in FILES:
    mid=source.name.split('_match_')[1].split('_ROLES.jsonl')[0]
    meta=json.loads((MATCH/mid/f'{mid}_match.json').read_text())
    signs=source_signs(meta)
    length=float(meta['pitch_length']);width=float(meta['pitch_width'])
    assert 90<=length<=120 and 45<=width<=90,(mid,length,width)
    hL,hW=length/2,width/2
    dest=OUT/f'P3_static_carry_corridors_match_{mid}.jsonl'
    counts=Counter();seen=set()
    with source.open(encoding='utf-8') as inp,dest.open('w',encoding='utf-8') as out:
        for line_no,line in enumerate(inp,1):
            if not line.strip():continue
            r=json.loads(line)
            key=(int(r['period']),int(r['frame']),str(r['possession_event_id']),str(r['carrier_player_id']))
            if key in seen:raise AssertionError(f'{mid}: duplicate decision key {key}')
            seen.add(key)
            counts['decisions']+=1
            c=finite_xy(r.get('carrier_xy'))
            if c is None:
                counts['missing_carrier_xy']+=1
                continue
            sign=signs[(int(r['period']),str(r['attacking_team_id']))]
            defenders=[];detected=[]
            for kind in ('detected','extrapolated'):
                for d in r.get(f'opponents_{kind}',[]):
                    q=finite_xy(d.get('xy'))
                    if q is not None:
                        defenders.append(q)
                        if kind=='detected':detected.append(q)
            counts['eligible_decisions']+=1
            for dist in LENGTHS:
                for angle in ANGLES:
                    rad=math.radians(angle)
                    # Angle signed in GLOBAL Y; positive and negative are mirror scenarios.
                    b=(c[0]+sign*dist*math.cos(rad),c[1]+dist*math.sin(rad))
                    inside=(-hL<=b[0]<=hL and -hW<=b[1]<=hW)
                    distances=[point_to_segment(q,c,b) for q in defenders]
                    direct=[point_to_segment(q,c,b) for q in detected]
                    obj={'match_id':mid,'period':key[0],'frame':key[1],
                        'possession_event_id':key[2],'carrier_player_id':key[3],
                        'attacking_team_id':str(r['attacking_team_id']),
                        'source_attack_x_sign':sign,'length_m':dist,'angle_from_attack_axis_deg':angle,
                        'signed_endpoint_dx_m':sign*(b[0]-c[0]),
                        'endpoint_within_pitch':inside,
                        'minimum_defender_segment_distance_m':min(distances) if distances else None,
                        'minimum_detected_defender_segment_distance_m':min(direct) if direct else None,
                        'number_finite_defenders':len(defenders),
                        'number_detected_defenders':len(detected),
                        'carrier_detected':r.get('carrier_detected') is True,
                        'interpretation':'hypothetical_static_carry_geometry_not_feasibility'}
                    out.write(json.dumps(obj,separators=(',',':'))+'\n')
                    counts['corridors']+=1
                    counts['out_of_pitch_endpoint']+=not inside
                    counts['minimum_defender_distance_defined']+=bool(distances)
                    counts['minimum_detected_defender_distance_defined']+=bool(direct)
    assert counts['corridors']==counts['eligible_decisions']*15,(mid,counts)
    TOTAL.update(counts)
    per_match.append({'match_id':mid,'pitch_length_m':length,'pitch_width_m':width,'counts':dict(counts)})
    print(mid,'decisions',counts['decisions'],'scenarios',counts['corridors'])
assert TOTAL['decisions']==18792,TOTAL
assert TOTAL['eligible_decisions']==18757,TOTAL
assert TOTAL['corridors']==18757*15,TOTAL
assert TOTAL['missing_carrier_xy']==35,TOTAL


1874553 decisions 959 scenarios 14340
1886347 decisions 999 scenarios 14985
1899585 decisions 929 scenarios 13935
1925299 decisions 1062 scenarios 15930
1927964 decisions 1140 scenarios 17085
1953632 decisions 979 scenarios 14550
1959846 decisions 946 scenarios 14175
1986691 decisions 915 scenarios 13725
1996435 decisions 1005 scenarios 14955
1996436 decisions 786 scenarios 11775
2006229 decisions 979 scenarios 14655
2006363 decisions 976 scenarios 14640
2007448 decisions 863 scenarios 12930
2007721 decisions 959 scenarios 14385
2010085 decisions 935 scenarios 14025
2011166 decisions 757 scenarios 11325
2013725 decisions 922 scenarios 13755
2015213 decisions 883 scenarios 13215
2016236 decisions 959 scenarios 14385
2017461 decisions 839 scenarios 12585


## Gate and limitations

There is deliberately **no binary carry-feasibility threshold**. Next steps should examine whether assumptions about carrier speed, ball control, defensive acceleration, and turning are scientifically supportable, and use matched decision-level summaries—not treat 15 scenarios as independent observations.


In [3]:
gate={'created_utc':datetime.now(timezone.utc).isoformat(),
  'purpose':'Hypothetical static carrying corridors at possession start; geometry only',
  'matches':20,'angles_deg':ANGLES,'lengths_m':LENGTHS,'totals':dict(TOTAL),
  'per_match':per_match,
  'invariants':{'decisions_18792':True,'eligible_18757':True,'corridors_per_eligible_decision':15,
                'source_metadata_orientation':True},
  'limitations':[
    'Hypothetical straight carrying displacements not actual carry opportunities or realized actions',
    'No event outcomes or future observations used to build scenarios',
    'No carrier speed, control, defender response, acceleration, turning or ball dynamics',
    'Pitch bounds use reported match pitch dimensions and center-origin coordinates',
    'Missing defender coordinates are not interpreted as open corridors',
    '15 scenarios per decision are dependent; match and possession clustering required',
    'Threshold-free minimum distance measures; not probabilistic interception or causal effects']}
p=OUT/'P3_static_carry_corridors_gate.json';p.write_text(json.dumps(gate,indent=2),encoding='utf-8')
print(json.dumps(dict(TOTAL),indent=2))
print('Upload:',p)


{
  "decisions": 18792,
  "eligible_decisions": 18757,
  "corridors": 281355,
  "out_of_pitch_endpoint": 15836,
  "minimum_defender_distance_defined": 281355,
  "minimum_detected_defender_distance_defined": 275295,
  "missing_carrier_xy": 35
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P3_static_carry_corridors_gate.json
